# 2장. 역전파: 넘파이로 신경망 구현하기

손실을 한 번 계산하는 것과 비슷한 비용으로 모든 가중치의 기울기를 구할 수 있는가.

**준비:** 한국전력거래소 「시간별 전국 전력수요량」 CSV 두 개(2023년, 2025년)와 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개(2023년, 2025년)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 2분쯤 걸린다.

## 2.1 연쇄 법칙으로 유닛 하나의 기울기 계산하기

In [ ]:
import numpy as np

# 자료 한 건과 유닛 하나
x, y = 2.0, 5.0
w, b, v, c = 0.5, 1.0, 1.5, 0.0

# 순전파: 입력에서 손실까지 차례로 계산하고, 중간 값을 모두 남겨 둔다
z = w * x + b
a = max(0.0, z)
out = v * a + c
L = (out - y) ** 2
print(z, a, out, L)

# 역전파: 손실에서 입력 쪽으로 가면서, 단계마다 그 단계의 미분을 곱한다
d_out = 2 * (out - y)                      # L = (out - y)²
d_v, d_c = d_out * a, d_out * 1            # out = v * a + c
d_a = d_out * v                            # out = v * a + c
d_z = d_a * (1.0 if z > 0 else 0.0)        # a = max(0, z)
d_w, d_b = d_z * x, d_z * 1                # z = w * x + b
print(d_out, d_v, d_c, d_a, d_z, d_w, d_b)

In [ ]:
def tiny_loss(w, b, v, c):
    return (v * max(0.0, w * x + b) + c - y) ** 2

eps = 1e-6
base = [w, b, v, c]
for k, name in enumerate(["w", "b", "v", "c"]):
    up, down = base.copy(), base.copy()
    up[k] += eps
    down[k] -= eps
    print(name, round((tiny_loss(*up) - tiny_loss(*down)) / (2 * eps), 6))

## 2.2 계산 그래프와 역전파의 계산 순서

In [ ]:
def one_day(x, y, w, b, v, c):
    """자료 한 건에 대한 순전파와 역전파. w, b, v는 길이 H인 배열이다."""
    z = w * x + b                    # (H,)
    a = np.maximum(0, z)             # (H,)
    out = a @ v + c                  # 숫자 하나
    L = (out - y) ** 2
    d_out = 2 * (out - y)
    d_v, d_c = d_out * a, d_out      # out = v1*a1 + v2*a2 + c
    d_a = d_out * v
    d_z = d_a * (z > 0)              # 꺼진 유닛으로는 기울기가 전달되지 않는다
    d_w, d_b = d_z * x, d_z
    return (z, a, out, L), (d_out, d_v, d_c, d_a, d_z, d_w, d_b)

def show(names, values):
    for name, val in zip(names, values):
        print(name, (np.round(val, 2) + 0.0).tolist())

# 1장에서 손으로 정한 신경망
w_h, b_h = np.array([-0.5, 0.5]), np.array([6.0, -10.0])
v_h, c_h = np.array([2.2, 3.0]), 62.0
values, cold = one_day(4.0, 74.0, w_h, b_h, v_h, c_h)      # 기온 4도, 실제 수요 74GW
show(["z", "a", "out", "L"], values)
show(["d_out", "d_v", "d_c", "d_a", "d_z", "d_w", "d_b"], cold)

In [ ]:
values, hot = one_day(30.0, 80.0, w_h, b_h, v_h, c_h)      # 기온 30도, 실제 수요 80GW
show(["z", "a", "out", "L"], values)
show(["d_w", "d_b", "d_v", "d_c"], [hot[5], hot[6], hot[1], hot[2]])

# 손실이 두 날의 평균이면 기울기도 두 날의 평균이다
both_days = [(g1 + g2) / 2 for g1, g2 in zip(cold, hot)]
mean_w, mean_b, mean_v, mean_c = both_days[5], both_days[6], both_days[1], both_days[2]
show(["d_w", "d_b", "d_v", "d_c"], [mean_w, mean_b, mean_v, mean_c])

## 2.3 행렬로 쓴 순전파와 역전파: 모양 확인하기

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

# 1장과 같다. 표준화에 쓰는 평균과 표준편차는 2023년에서만 구한다
t_mean, t_std = T23.mean(), T23.std()
y_mean, y_std = Y23.mean(), Y23.std()
xs, ys = (T23 - t_mean) / t_std, (Y23 - y_mean) / y_std

def unpack(theta):
    """theta를 w, b, v(각각 길이 H)와 c로 나눈다. theta의 길이는 3H + 1이다."""
    H = (len(theta) - 1) // 3
    return theta[:H], theta[H:2 * H], theta[2 * H:3 * H], theta[3 * H]

def forward(theta, x):
    """순전파. 예측 (N,)과, 역전파에 쓸 중간 값 z, a(각각 (N, H))를 돌려준다."""
    w, b, v, c = unpack(theta)
    z = np.outer(x, w) + b
    a = np.maximum(0, z)
    return a @ v + c, z, a

def loss(theta):
    return float(((forward(theta, xs)[0] - ys) ** 2).mean())

def numeric(theta, eps=1e-5):
    """1장의 수치 미분. 숫자 하나마다 손실을 두 번 계산한다."""
    g = np.zeros(len(theta))
    for i in range(len(theta)):
        move = np.zeros(len(theta))
        move[i] = eps
        g[i] = (loss(theta + move) - loss(theta - move)) / (2 * eps)
    return g

def start(seed):
    return np.append(np.random.default_rng(seed).normal(0, 1, 6), 0.0)

theta0 = start(3)
print(round(loss(theta0), 4), numeric(theta0).round(3).tolist())

In [ ]:
def backward(theta, x=xs, y=ys):
    """역전파. 손실을 theta의 숫자마다 미분한 값을 theta와 같은 순서로 돌려준다."""
    w, b, v, c = unpack(theta)
    out, z, a = forward(theta, x)
    d_out = 2 * (out - y) / len(x)      # (N,)    손실은 N건의 평균이다
    d_v = a.T @ d_out                   # (H,)
    d_c = d_out.sum()                   # 숫자 하나
    d_a = np.outer(d_out, v)            # (N, H)
    d_z = d_a * (z > 0)                 # (N, H)
    d_w = x @ d_z                       # (H,)
    d_b = d_z.sum(axis=0)               # (H,)
    return np.concatenate([d_w, d_b, d_v, [d_c]])

out, z, a = forward(theta0, xs)
print(out.shape, z.shape, a.shape, backward(theta0).shape)
print(backward(theta0).round(3).tolist())
print(f"{np.abs(backward(theta0) - numeric(theta0)).max():.1e}")

## 2.4 수치 미분으로 기울기 검증하기

In [ ]:
def gap(g1, g2):
    """두 기울기의 상대 오차. 같으면 0이고, 방향이 반대이면 1이다."""
    return float(np.linalg.norm(g1 - g2) / (np.linalg.norm(g1) + np.linalg.norm(g2)))

exact = backward(theta0)
for eps in [1e-1, 1e-2, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11, 1e-13]:
    print(f"{eps:.0e}  {gap(exact, numeric(theta0, eps)):.1e}")

In [ ]:
def crossed(theta, eps):
    """숫자를 하나씩 eps만큼 늘리고 줄였을 때 켜짐과 꺼짐이 달라지는 (날, 유닛)의 수."""
    count = 0
    for i in range(len(theta)):
        move = np.zeros(len(theta))
        move[i] = eps
        on_up = forward(theta + move, xs)[1] > 0
        on_down = forward(theta - move, xs)[1] > 0
        count += int((on_up != on_down).sum())
    return count

print(crossed(theta0, 1e-1), crossed(theta0, 1e-2), crossed(theta0, 1e-3))

In [ ]:
# 유닛 1의 편향을 고쳐서, 2023년의 101번째 날이 정확히 꺾이는 점 위에 오게 한다
on_kink = theta0.copy()
on_kink[2] = -(xs[100] * on_kink[0])
z = forward(on_kink, xs)[1]
print(float(z[100, 0]), int((z == 0).sum()))

by_back, by_numeric = backward(on_kink), numeric(on_kink)
print(by_back.round(5).tolist())
print(by_numeric.round(5).tolist())
print(f"{gap(by_back, by_numeric):.1e}")

# 유닛 1의 편향(theta의 세 번째 숫자)만 보고, 오른쪽 기울기와 왼쪽 기울기를 따로 구한다
move = np.zeros(7)
move[2] = 1e-6
right = (loss(on_kink + move) - loss(on_kink)) / 1e-6
left = (loss(on_kink) - loss(on_kink - move)) / 1e-6
middle = (left + right) / 2
print(round(left, 5), round(right, 5), round(middle, 5), round(float(by_back[2]), 5))

In [ ]:
def wrong(theta, kind, x=xs, y=ys):
    """일부러 한 군데를 틀리게 쓴 역전파. kind가 틀린 곳이다."""
    w, b, v, c = unpack(theta)
    out, z, a = forward(theta, x)
    d_out = (1 if kind == "2를 빠뜨림" else 2) * (out - y) / len(x)
    d_a = np.outer(d_out, np.ones_like(v) if kind == "v를 빠뜨림" else v)
    d_z = d_a if kind == "ReLU 조건을 빠뜨림" else d_a * (z > 0)
    d_b = d_z.mean(axis=0) if kind == "합 대신 평균" else d_z.sum(axis=0)
    return np.concatenate([x @ d_z, d_b, a.T @ d_out, [d_out.sum()]])

def descend(grad, theta, steps=5000, rate=0.05):
    """grad로 기울기를 구해 경사하강법을 한다. 마지막 theta와 손실의 기록을 돌려준다."""
    trace = [loss(theta)]
    for _ in range(steps):
        theta = theta - rate * grad(theta)
        trace.append(loss(theta))
    return theta, np.array(trace)

def gw(theta, t):
    return forward(theta, (t - t_mean) / t_std)[0] * y_std + y_mean

kinds = ["틀린 곳 없음", "2를 빠뜨림", "v를 빠뜨림", "ReLU 조건을 빠뜨림",
         "합 대신 평균"]
traces = {}
for kind in kinds:
    for seed in [3, 0]:
        grad = lambda theta: wrong(theta, kind)
        end, traces[kind, seed] = descend(grad, start(seed))
        trace = traces[kind, seed]
        error = gap(grad(start(seed)), numeric(start(seed)))
        losses = f"{trace[0]:.4f} → {trace[100]:.4f} → {trace[-1]:.4f}"
        test_rmse = rmse(Y25, gw(end, T25))
        print(f"{kind:<12} 시드 {seed}  {error:.2f}  {losses}  {test_rmse:.2f}")

## 2.5 넘파이 신경망으로 1장의 결과 다시 얻기

In [ ]:
import time

began = time.perf_counter()
runs = [descend(backward, start(seed))[0] for seed in range(10)]
t_back = time.perf_counter() - began

def kinks(theta):
    w, b, v, c = unpack(theta)
    return -b / w * t_std + t_mean

for seed, theta in enumerate(runs):
    print(seed, f"{loss(theta):.4f}", f"{rmse(Y23, gw(theta, T23)):.2f}",
          f"{rmse(Y25, gw(theta, T25)):.2f}", kinks(theta).round(1).tolist())

began = time.perf_counter()
by_numeric, _ = descend(numeric, start(3))
t_numeric = time.perf_counter() - began
print(by_numeric.round(2).tolist())
print(runs[3].round(2).tolist())
print(f"{np.abs(by_numeric - runs[3]).max():.4f}")

In [ ]:
print(f"역전파로 시드 하나: {t_back / 10:.2f}초")
print(f"수치 미분으로 시드 하나: {t_numeric:.2f}초")

In [ ]:
def clock(run, repeat):
    """run을 repeat번 실행해서 한 번에 걸린 시간을 밀리초로 돌려준다."""
    began = time.perf_counter()
    for _ in range(repeat):
        run()
    return (time.perf_counter() - began) / repeat * 1000

timing = []
for H in [2, 8, 32, 128, 512, 2048]:
    theta = np.append(np.random.default_rng(0).normal(0, 1, 3 * H), 0.0)
    t_loss = clock(lambda: loss(theta), 300)
    t_bp = clock(lambda: backward(theta), 300)
    t_num = clock(lambda: numeric(theta), 3 if H <= 128 else 1)
    timing.append((H, 3 * H + 1, t_loss, t_bp, t_num))
    print(f"{H:>5} {3 * H + 1:>5}  {t_loss:8.3f}  {t_bp:8.3f}  {t_num:10.1f}"
          f"  {t_num / t_bp:7.0f}배  {t_bp / t_loss:4.1f}배")

In [ ]:
starts = [("모두 0", np.zeros(7)), ("모두 0.5", np.full(7, 0.5)),
          ("난수(시드 3)", start(3))]
for name, theta in starts:
    first = backward(theta)
    end, _ = descend(backward, theta)
    test_rmse = rmse(Y25, gw(end, T25))
    print(name)
    print("  처음 기울기", (first.round(3) + 0.0).tolist())
    after = (end.round(2) + 0.0).tolist()
    print("  학습 뒤", after, f"{loss(end):.4f}", f"{test_rmse:.2f}")

In [ ]:
def init_layers(sizes, seed):
    """sizes = [입력 수, 은닉층의 유닛 수들, 1]. 층마다 가중치 (m, k)와 편향 (k,)."""
    rng = np.random.default_rng(seed)
    return [(rng.normal(0, 1 / np.sqrt(m), (m, k)), np.zeros(k))
            for m, k in zip(sizes[:-1], sizes[1:])]

def forward_all(layers, X):
    """X: (N, D). 층마다 그 층의 입력 A와 Z를 남겨 둔다. 마지막 층에는 ReLU가 없다."""
    A, kept = X, []
    for i, (W, b) in enumerate(layers):
        Z = A @ W + b
        kept.append((A, Z))
        A = Z if i == len(layers) - 1 else np.maximum(0, Z)
    return A[:, 0], kept

def backward_all(layers, X, y):
    """마지막 층부터 거꾸로 가면서 층마다 (가중치의 기울기, 편향의 기울기)를 구한다."""
    out, kept = forward_all(layers, X)
    d = (2 * (out - y) / len(y))[:, None]          # (N, 1)
    grads = []
    for i in reversed(range(len(layers))):
        A, Z = kept[i]
        if i < len(layers) - 1:
            d = d * (Z > 0)                        # 이 층의 ReLU를 거꾸로 지난다
        grads.append((A.T @ d, d.sum(axis=0)))     # 가중치 (m, k), 편향 (k,)
        d = d @ layers[i][0].T                     # 앞 층의 출력에 대한 기울기 (N, m)
    return grads[::-1]

X23 = xs[:, None]                                  # (247, 1)
layers = init_layers([1, 8, 8, 1], seed=0)
grads = backward_all(layers, X23, ys)
print([W.shape for W, b in layers], [b.shape for W, b in layers])
print([gW.shape for gW, gb in grads], [gb.shape for gW, gb in grads])
print(sum(W.size + b.size for W, b in layers))

In [ ]:
def loss_all(layers, X, y):
    return float(((forward_all(layers, X)[0] - y) ** 2).mean())

def numeric_all(layers, X, y, eps=1e-5):
    """모든 가중치와 편향을 하나씩 바꿔 가며 수치 미분한다. 검증에만 쓴다."""
    found = []
    for W, b in layers:
        for arr in (W, b):
            g = np.zeros_like(arr)
            for idx in np.ndindex(arr.shape):
                keep = arr[idx]
                arr[idx] = keep + eps
                up = loss_all(layers, X, y)
                arr[idx] = keep - eps
                down = loss_all(layers, X, y)
                arr[idx] = keep
                g[idx] = (up - down) / (2 * eps)
            found.append(g.ravel())
    return np.concatenate(found)

flat = np.concatenate([g.ravel() for pair in grads for g in pair])
print(flat.shape, f"{gap(flat, numeric_all(layers, X23, ys)):.1e}")

In [ ]:
def train_layers(sizes, seed, steps=20000, rate=0.05):
    layers = init_layers(sizes, seed)
    for _ in range(steps):
        grads = backward_all(layers, X23, ys)
        layers = [(W - rate * gW, b - rate * gb)
                  for (W, b), (gW, gb) in zip(layers, grads)]
    return layers

def gw_all(layers, t):
    return forward_all(layers, ((t - t_mean) / t_std)[:, None])[0] * y_std + y_mean

for sizes in [[1, 8, 1], [1, 8, 8, 1]]:
    marks = []
    for seed in range(5):
        layers = train_layers(sizes, seed)
        marks.append((rmse(Y23, gw_all(layers, T23)), rmse(Y25, gw_all(layers, T25))))
    marks = np.array(marks)
    train, test = np.sort(marks[:, 0]).round(2), np.sort(marks[:, 1]).round(2)
    print(sizes, train.tolist(), test.tolist())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 2-5
sweep = np.logspace(-14, 0, 57)
errors = np.array([gap(exact, numeric(theta0, e)) for e in sweep])
best = int(errors.argmin())
sci = FuncFormatter(lambda val, _: f"{val:.0e}")
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.plot(sweep[sweep <= 1e-2], 1e-16 / sweep[sweep <= 1e-2], color=GRAY, lw=1.2, ls=(0, (5, 2)), label="1e-16 ÷ ε")
ax.plot(sweep, np.maximum(errors, 1e-17), color=BRICK, lw=1.8, marker="o", ms=3.2, label="역전파와 수치 미분의 상대 오차")
ax.scatter([sweep[best]], [errors[best]], s=46, color=TEAL, zorder=5, edgecolor="white", lw=0.8)
ax.scatter([1e-5], [gap(exact, numeric(theta0, 1e-5))], s=46, color=NAVY, zorder=5, edgecolor="white", lw=0.8)
ax.text(sweep[best] * 0.75, errors[best] * 0.22, f"가장 작은 오차\nε = {sweep[best]:.0e}, {errors[best]:.0e}", fontsize=9, color=TEAL, ha="right", va="center", linespacing=1.5)
ax.text(6e-6, 4e-14, "이 책에서 쓰는 ε = 1e-05\n오차 4e-12", fontsize=9, color=NAVY, ha="right", va="center", linespacing=1.5)
ax.text(3e-14, 2e-7, "ε이 작아질수록\n반올림 오차가 커진다", fontsize=9, color=BRICK, ha="left", va="center", linespacing=1.5)
ax.text(2.2e-2, 2e-7, "ε이 크면 켜짐과 꺼짐이\n바뀌는 날이 생긴다", fontsize=9, color=BRICK, ha="right", va="center", linespacing=1.5)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(5e-15, 2); ax.set_ylim(1e-17, 1e-1)
ax.set_xticks([1e-14, 1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1]); ax.set_yticks([1e-16, 1e-14, 1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2])
ax.xaxis.set_major_formatter(sci); ax.yaxis.set_major_formatter(sci)
ax.minorticks_off()
ax.set_xlabel("수치 미분의 간격 ε (로그 축)", fontsize=9)
ax.set_ylabel("상대 오차 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_2_5.png", dpi=300)
plt.show()

In [ ]:
# 그림 2-6
shifts = np.linspace(-0.004, 0.004, 401)
slope = []
for s in shifts:
    moved = on_kink.copy()
    moved[2] += s
    slope.append(backward(moved)[2])
slope = np.array(slope)
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.plot(shifts[shifts < 0], slope[shifts < 0], color=NAVY, lw=1.8)
ax.plot(shifts[shifts > 0], slope[shifts > 0], color=NAVY, lw=1.8)
ax.axvline(0, color=GRAY, lw=0.8, ls=(0, (2, 2)))
ax.scatter([0, 0], [left, right], s=40, color=[BRICK, BRICK], zorder=5, edgecolor="white", lw=0.8)
ax.scatter([0], [(left + right) / 2], s=40, color=TEAL, zorder=6, edgecolor="white", lw=0.8, marker="D")
ax.text(-0.00018, left - 0.0005, f"왼쪽 기울기 {left:.5f}\n역전파의 값과 같다", fontsize=9, color=BRICK, ha="right", va="top", linespacing=1.5)
ax.text(0.00018, right + 0.0005, f"오른쪽 기울기 {right:.5f}", fontsize=9, color=BRICK, ha="left", va="bottom")
ax.text(0.00015, (left + right) / 2, f"수치 미분의 값 {(left + right) / 2:.5f}\n두 기울기의 평균", fontsize=9, color=TEAL, ha="left", va="center", linespacing=1.5)
ax.set_xlim(-0.0042, 0.0042); ax.set_ylim(right - 0.0006, left + 0.0006)
ax.set_xlabel("유닛 1의 편향을 옮긴 크기 (0이면 101번째 날이 꺾이는 점 위에 있다)", fontsize=9)
ax.set_ylabel("손실을 유닛 1의 편향으로 미분한 값", fontsize=9)
frame(ax)
fig.tight_layout()
fig.savefig("fig_2_6.png", dpi=300)
plt.show()

In [ ]:
# 그림 2-7
looks = {"틀린 곳 없음": (TEAL, "-", 2.6), "2를 빠뜨림": (NAVY, (0, (5, 2)), 1.4), "v를 빠뜨림": (NAVY, (0, (1.5, 1.5)), 1.4),
         "ReLU 조건을 빠뜨림": (BRICK, "-", 1.6), "합 대신 평균": (BRICK, (0, (5, 2)), 1.6)}
fig, ax = plt.subplots(figsize=(7, 4.8))
steps = np.arange(1, 5001)
for kind, (col, ls, lw) in looks.items():
    trace = traces[kind, 3]
    ax.plot(steps, trace[1:], color=col, ls=ls, lw=lw, label=f"{kind}: 5,000걸음 뒤 {trace[-1]:.4f}")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(0.9, 5500); ax.set_ylim(0.18, 2.6)
plain_ticks(ax.xaxis)
ax.set_yticks([0.2, 0.3, 0.5, 1, 2]); ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:g}"))
ax.minorticks_off()
ax.set_xlabel("경사하강법의 걸음 수(로그 축)", fontsize=9)
ax.set_ylabel("손실(표준화한 단위, 로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_2_7.png", dpi=300)
plt.show()

In [ ]:
# 그림 2-8
counts = [row[1] for row in timing]
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.plot(counts, [row[4] for row in timing], color=BRICK, lw=1.8, marker="o", ms=5, label="수치 미분으로 기울기 한 번")
ax.plot(counts, [row[3] for row in timing], color=TEAL, lw=2.2, marker="o", ms=5, label="역전파로 기울기 한 번")
ax.plot(counts, [row[2] for row in timing], color=GRAY, lw=1.3, ls=(0, (5, 2)), marker="s", ms=4, label="손실 한 번 계산")
for row, dx, dy, ha in [(timing[0], 1.0, 2.6, "center"), (timing[-1], 0.72, 1.6, "right")]:
    ax.text(row[1] * dx, row[4] * dy * 1.5, f"{row[4]:,.1f}밀리초\n역전파의 {row[4] / row[3]:,.0f}배", fontsize=9, color=BRICK, ha=ha, va="center",
            weight="bold", linespacing=1.4)
    ax.text(row[1] * dx, row[3] * 2.6, f"{row[3]:.3f}밀리초", fontsize=9, color=TEAL, ha=ha, va="center", weight="bold")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(4, 12000); ax.set_ylim(0.005, 200000)
ax.set_xticks(counts); ax.set_xticklabels([f"{n:,}" for n in counts])
ax.set_yticks([0.01, 0.1, 1, 10, 100, 1000, 10000, 100000])
ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:g}"))
ax.minorticks_off()
ax.set_xlabel("가중치와 편향의 수(로그 축)", fontsize=9)
ax.set_ylabel("걸린 시간(밀리초, 로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_2_8.png", dpi=300)
plt.show()

In [ ]:
# 그림 2-9
grid_t = np.linspace(-10, 31, 411)
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.scatter(T23, Y23, s=10, color=to_rgba(GRAY, 0.6), edgecolor="none")
for name, theta, col, ls, lw in [("모두 0에서 시작", np.zeros(7), BRICK, (0, (5, 2)), 1.6), ("모두 0.5에서 시작", np.full(7, 0.5), NAVY, "-", 1.6),
                                 ("난수(시드 3)에서 시작", start(3), TEAL, "-", 2.4)]:
    end, _ = descend(backward, theta)
    ax.plot(grid_t, gw(end, grid_t), color=col, ls=ls, lw=lw, label=f"{name}: 2025년 RMSE {rmse(Y25, gw(end, T25)):.2f}GW")
ax.set_xlim(-10.5, 31.5); ax.set_ylim(54, 94); ax.set_xticks(range(-10, 31, 5))
ax.set_xlabel("일평균 기온(°C)", fontsize=9)
ax.set_ylabel("일평균 전력수요(GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_2_9.png", dpi=300)
plt.show()